In [ ]:
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, '..')

from pathlib import Path

# Ensure output directory exists before saving any charts
Path('../evaluation/results').mkdir(parents=True, exist_ok=True)

with open('../evaluation/results/ablation_results.json') as f:
    ablation = json.load(f)

df = pd.DataFrame(ablation)

# Shared color palette — one color per strategy, used across all charts
STRATEGY_COLORS = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']

print(df[['strategy', 'faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']].to_string(index=False))

In [ ]:
metrics = ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']
metric_labels = ['Faithfulness', 'Answer\nRelevancy', 'Context\nPrecision', 'Context\nRecall']

N = len(metrics)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for row, color in zip(df.itertuples(), STRATEGY_COLORS):
    values = [getattr(row, m, 0) for m in metrics]
    values += values[:1]
    ax.plot(angles, values, 'o-', color=color, linewidth=2, label=row.strategy)
    ax.fill(angles, values, alpha=0.15, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metric_labels, size=12)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], size=8)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=11)
ax.set_title('RAG Chunking Strategy Comparison', size=14, pad=20)

plt.tight_layout()
plt.savefig('../evaluation/results/radar_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to evaluation/results/radar_chart.png')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(len(df))
width = 0.2

ax = axes[0]
for i, metric in enumerate(metrics):
    ax.bar(x + i * width, df[metric], width, label=metric.replace('_', ' ').title(), alpha=0.85)
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels([s.title() for s in df['strategy']])
ax.set_ylabel('RAGAS Score')
ax.set_title('RAGAS Metrics by Chunking Strategy')
ax.legend()
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)

ax2 = axes[1]
if 'avg_answer_latency_ms' in df.columns:
    bars = ax2.bar(df['strategy'], df['avg_answer_latency_ms'], color=STRATEGY_COLORS)
    ax2.set_ylabel('Average Latency (ms)')
    ax2.set_title('Answer Generation Latency')
    ax2.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, df['avg_answer_latency_ms']):
        ax2.text(
            bar.get_x() + bar.get_width() / 2.,
            bar.get_height() + 10,
            f'{val:.0f}ms',
            ha='center', va='bottom', fontsize=10,
        )

plt.tight_layout()
plt.savefig('../evaluation/results/metrics_comparison.png', dpi=150)
plt.show()
print('Saved to evaluation/results/metrics_comparison.png')

# Print markdown table for README
print('\n### README-ready table:\n')
print('| Strategy | Faithfulness | Answer Relevancy | Context Precision | Context Recall | Avg Latency |')
print('|---|---|---|---|---|---|')
for _, row in df.iterrows():
    faithfulness      = row['faithfulness'] if isinstance(row['faithfulness'], float) else 0.0
    answer_relevancy  = row['answer_relevancy'] if isinstance(row['answer_relevancy'], float) else 0.0
    context_precision = row['context_precision'] if isinstance(row['context_precision'], float) else 0.0
    context_recall    = row['context_recall'] if isinstance(row['context_recall'], float) else 0.0
    latency           = row.get('avg_answer_latency_ms', 0)
    print(
        f"| {row['strategy'].title()} "
        f"| {faithfulness:.3f} "
        f"| {answer_relevancy:.3f} "
        f"| {context_precision:.3f} "
        f"| {context_recall:.3f} "
        f"| {latency:.0f}ms |"
    )